# BT4012 Kaggle Competition: Detecting Illicit Bitcoin Transactions
**Student number:** e1156682

This notebook builds my final submission (public leaderboard AUC **0.96169**): a **two-stage (stacked) LightGBM** that scores each test transaction from its own features, the profile of its **present neighbours** in the payment graph and the **predicted risk of those neighbours**. It is trained with a **per-step offset** so that the illicit base rates of the training steps do not carry into the test period. The notebook runs end to end, from the raw Kaggle files to the submission CSV, without importing anything from `src/`.

## The task
Each row is a Bitcoin transaction with 165 anonymised features and a time step (1–49, roughly two weeks apart). Training transactions are labeled **illicit** (1), **licit** (0) or unknown; most are unknown.
The edge list records the payment flow: an edge `txId1 → txId2` means an output of `txId1` is spent by `txId2`.
The target is the probability that each test transaction is illicit, scored by **ROC-AUC**.

The test set covers **later** time steps (36–49) than the training set (1–35), so the central difficulty is temporal drift rather than separating the classes within the training period.

## Result
| Model | What it adds | Public leaderboard AUC |
|---|---|---|
| Baseline | LightGBM on the 165 raw features | 0.94637 |
| + graph features | 27 features describing each transaction's neighbours in the payment graph | 0.95121 |
| + neighbours' predicted risk | A second LightGBM that also sees how suspicious the neighbours look (two-stage, stacked model) | 0.95846 |
| **Final model** | + a per-step offset and no class weight, averaged over 6 random seeds | **0.96169** |

The first two additions also raised the AUC on every validation fold. The last one ties on validation, so I selected it on the public leaderboard (sections 6 and 8).

## Approach
1. **Present-neighbour graph features** (section 5.2): summaries of each transaction's neighbours, counting a neighbour only if it is **present** (a labeled training row or a `test.csv` row). Training and test rows then observe their neighbourhoods under the same conditions (section 5.1).
2. **Neighbours' predicted risk** (section 5.3): a stage-1 LightGBM scores every present transaction out of fold. The mean and max score over each transaction's present neighbours become two inputs to a stage-2 LightGBM.
3. **Per-step offset, no class weight** (section 4): each training row starts from its time step's illicit log-odds, relative to the overall rate. The trees then learn what separates illicit from licit transactions *within* a step, instead of absorbing each training step's base rate.

## Contents
| Section | What it does | Part of the brief |
|---|---|---|
| 1. Setup | Imports, seeds, paths | |
| 2. Load the data | Reads the four Kaggle files | |
| 3. Preprocessing | Data checks; selects the labeled training rows | Data preprocessing |
| 4. The learner | LightGBM with a per-step offset and no class weight | Model |
| 5. Feature engineering | Present rule, 27 graph features, 2 neighbours' risk features | Feature engineering |
| 6. Validation | Four late time-based folds, each rebuilt from its own training period | Prediction validation |
| 7. Final model and submission | Averages 6 seeds and writes the CSV | Generating the submission CSV |
| 8. Summary | Leaderboard history, final picks, ensembles, limitations | |

Section 4 comes before feature engineering because the neighbours' risk features (section 5.3) are built with that learner.

## How to run
1. Put the four Kaggle files (`train.csv`, `test.csv`, `txs_edgelist.csv`, `sample_submission.csv`) in `data/raw/`.
2. `pip install -r requirements.txt` (tested with Python 3.14.6, pandas 3.0.6, numpy 2.5.3, scipy 1.18.1, scikit-learn 1.9.1 and lightgbm 4.7.0).
3. Run all cells (about 7 minutes on my PC, with the 2 LightGBM threads set in section 1). The submission is written to `submissions/e1156682_submission.csv`, and the last code cell prints the total run time.

`notebooks/01_eda.ipynb` to `notebooks/05_model_comparison.ipynb` contain the exploration behind each decision; this notebook keeps only what the final model needs.

## 1. Setup
All imports in one place; every package is pinned in `requirements.txt`.
The cell prints the package versions, because LightGBM and scikit-learn results can change in the last digits between versions.

| Library | Used for |
|---|---|
| `lightgbm` | The learner in both stages (section 4) |
| `scipy.sparse` | The payment graph as sparse adjacency matrices (section 5.2) |
| `scikit-learn` | PCA of the local features, `GroupKFold` for the out-of-fold scores, ROC-AUC |
| `pandas`, `numpy` | Data handling |

In [1]:
import platform
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.sparse import coo_matrix
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

print(f"Python {platform.python_version()}, pandas {pd.__version__}, numpy {np.__version__}, scipy {scipy.__version__}, "
      f"scikit-learn {sklearn.__version__}, lightgbm {lgb.__version__}")

Python 3.14.6, pandas 3.0.6, numpy 2.5.3, scipy 1.18.1, scikit-learn 1.9.1, lightgbm 4.7.0


**Settings used throughout.** Fixed seeds and a fixed thread count make every run reproducible on the same machine and package versions: LightGBM's floating-point sums can change in the last digits when the number of threads changes.
`N_THREADS = 2` is the setting the leaderboard file was produced with. More threads run faster, but the predictions may then differ in the last digits.

In [2]:
START_TIME = time.time()                # to report the total run time at the end

SEED = 42                               # seed for the validation runs
FINAL_SEEDS = [42, 43, 44, 45, 46, 47]  # the submission averages six models that differ only in their seed
N_THREADS = 2                           # LightGBM threads, kept fixed (see above)
DATA_DIR = Path("data/raw")             # the four Kaggle CSVs (git-ignored)
SUBMISSIONS_DIR = Path("submissions")

ID_COL, TIME_COL, LABEL_COL = "txId", "time_step", "label"
FEATURE_COLS = [f"feat_{i}" for i in range(1, 166)]  # 165 anonymised, already normalised features
LOCAL_COLS = FEATURE_COLS[:93]  # the first 93 describe the transaction itself; most of the rest summarise its neighbours

## 2. Load the data
The three data files, plus Kaggle's sample submission, which fixes the row order of the CSV written in section 7.

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")           # steps 1-35; label is 1, 0, or blank (unknown)
test = pd.read_csv(DATA_DIR / "test.csv")             # steps 36-49; no label, plus an `index` column for the submission
edges = pd.read_csv(DATA_DIR / "txs_edgelist.csv")    # txId1 -> txId2: an output of txId1 is spent by txId2
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"train: {train.shape[0]:,} rows, steps {train[TIME_COL].min()}-{train[TIME_COL].max()}")
print(f"test:  {test.shape[0]:,} rows, steps {test[TIME_COL].min()}-{test[TIME_COL].max()}")
print(f"edges: {edges.shape[0]:,}")
train[LABEL_COL].map({1: "illicit", 0: "licit"}).fillna("unknown").value_counts()

train: 141,772 rows, steps 1-35
test:  15,329 rows, steps 36-49
edges: 234,355


label
unknown    110537
licit       27591
illicit      3644
Name: count, dtype: int64

**Takeaway:** Only 22% of the training rows (31,235 of 141,772) carry a label, and 11.7% of those (3,644) are illicit.
The 110,537 unlabeled rows cannot train the model, but their feature values still contribute to the PCA in section 5.2.

## 3. Preprocessing
The features arrive already normalised, and tree ensembles depend only on the order of feature values, so preprocessing comes down to validating the data and selecting the training rows.

**Checks in the next cell:**
- **Missing values and duplicate ids:** is there anything to impute or de-duplicate?
- **Train–test edges:** can a test transaction have a training neighbour, and so a known label nearby?
- **Cross-step edges:** do linked transactions always share a time step? Section 5.3 relies on this.
- **Edge endpoints without a row:** how much of the graph has no features at all?

In [4]:
# Data checks: nothing to impute or de-duplicate, and how the edges connect transactions
all_ids = pd.concat([train[ID_COL], test[ID_COL]])
step_of = pd.concat([train, test]).set_index(ID_COL)[TIME_COL]           # time step of every transaction with a row
step1, step2 = edges["txId1"].map(step_of), edges["txId2"].map(step_of)  # NaN = endpoint has no row
crosses = (edges["txId1"].isin(train[ID_COL]) & edges["txId2"].isin(test[ID_COL])) | \
          (edges["txId1"].isin(test[ID_COL]) & edges["txId2"].isin(train[ID_COL]))
print("Missing feature values:", int(train[FEATURE_COLS].isna().sum().sum() + test[FEATURE_COLS].isna().sum().sum()))
print("Duplicate transaction ids:", int(all_ids.duplicated().sum()))
print("Edges between a training and a test transaction:", int(crosses.sum()))
print("Edges between two different time steps (both ends have a row):", int((step1.notna() & step2.notna() & (step1 != step2)).sum()))
print("Edge endpoints with no row in train.csv or test.csv:", int((~pd.concat([edges["txId1"], edges["txId2"]]).isin(all_ids)).sum()))

Missing feature values: 0
Duplicate transaction ids: 0
Edges between a training and a test transaction: 0
Edges between two different time steps (both ends have a row): 0
Edge endpoints with no row in train.csv or test.csv: 92629


**Takeaway:** Nothing needs imputing or de-duplicating, and no edge joins a training and a test transaction.
Every edge between two transactions that have a row stays inside one time step, so a transaction's present neighbours (section 5.1) always share its step; section 5.3 relies on this.
Transactions without a row have no known step, so their edges cannot be checked.

The last number shapes the feature engineering (section 5). `test.csv` appears to contain only the **labeled** transactions of steps 36–49
(I infer this from the data; the neighbour shares in section 5.1 are consistent with it), so the unlabeled test-period transactions appear in the edge list but have no features.

A supervised model can only learn from, and be scored on, labeled rows, so I train on the labeled training rows.
The unlabeled training rows are not discarded: their feature values are used to fit the PCA in section 5.2.

In [5]:
lab = train[train[LABEL_COL].notna()].reset_index(drop=True)  # lab = the labeled training rows
lab[LABEL_COL] = lab[LABEL_COL].astype(int)
print(f"Labeled training rows: {len(lab):,}, of which {lab[LABEL_COL].mean():.1%} illicit")

share = lab.groupby(TIME_COL)[LABEL_COL].mean()
print(f"Illicit share per time step: from {share.min():.1%} (step {share.idxmin()}) to {share.max():.1%} (step {share.idxmax()})")

Labeled training rows: 31,235, of which 11.7% illicit
Illicit share per time step: from 0.4% (step 5) to 36.0% (step 13)


**Takeaway:** The illicit share moves from 0.4% to 36.0% between time steps. This has two consequences: validation must respect time order (section 6), and the model should not lean on a step's average illicit rate (section 4).

## 4. The learner: LightGBM with a per-step offset and no class weight
Every model in this notebook is a LightGBM with one fixed set of hyperparameters, taken from `notebooks/03_baseline_models.ipynb`.
Gradient-boosted trees suit this data: the features are heavily correlated and long-tailed, the graph features in section 5 contain missing values, and the decision boundary is non-linear (logistic regression trails LightGBM by about 0.09 AUC in notebook 03).
The one tuning I submitted (1000 trees, 30% of the features per tree) scored lower on the leaderboard, both without and with the graph features (0.94261 and 0.94802; section 8), so the settings stay fixed.

**The problem.** The illicit share per training step ranges from 0.4% to 36.0% (section 3). Many features drift over time, so a tree model can recognise which step a transaction comes from
and score it partly by *which step it resembles*. The test steps 36–49 have their own, unknown illicit shares, so that part of the score does not transfer.

**Design choices**
1. **Per-step offset.** LightGBM accepts a fixed starting score for each row (`init_score`) instead of one shared starting value.
   I set it to the log-odds of the row's step illicit rate minus the log-odds of the overall rate: 0 for a step at the overall rate, negative for a less illicit step, positive for a more illicit one.
   Each step's base rate is then absorbed by the offset, so the trees concentrate on what separates illicit from licit transactions *within* a step.
   LightGBM does not add `init_score` at prediction time, so no training step's base rate enters the test scores.
   Rates are smoothed as (illicit rows + 0.5) / (rows + 1), which keeps the log-odds finite even for a step with no illicit rows.
2. **No class weight.** My earlier models used `class_weight="balanced"`, which up-weights each illicit row (by a factor of 7.6, printed below).
   That factor is the same in every step, so it shifts every step's log-odds equally and leaves the differences between steps, which the offset describes, unchanged.
   What it does change is how strongly the illicit rows pull on the splits. Without it, every row counts equally.
3. **How they were chosen.** Both choices were selected on the public leaderboard. On my validation folds the final model ties with the model without them (section 6, discussed in section 8).

**Worked example.** Step 13 has 291 illicit rows out of 809, so its smoothed rate is (291 + 0.5) / 810 = 0.360. The overall rate is (3,644 + 0.5) / 31,236 = 0.117.
The offset is logit(0.360) − logit(0.117) = −0.576 − (−2.024) = **+1.448**: every step-13 row starts 1.448 log-odds above an average row before the first tree is built.
Step 5 (8 illicit rows out of 1,882) gets **−3.372**.

**Helpers**
- `logit(p)`: log-odds of a probability.
- `step_offset(rows)`: the starting score of each row, computed from the labels of `rows` only (always training rows).
- `make_lightgbm(seed, balanced)`: an untrained LightGBM with the fixed settings.
- `fit_lightgbm(rows, cols, seed, balanced, offset)`: fits that LightGBM on `rows[cols]`, starting from `offset`.

In [6]:
def logit(p):
    """Log-odds of a probability."""
    return np.log(p / (1 - p))


def step_offset(rows):
    """LightGBM starting score for each row: logit(smoothed illicit rate of the row's time step) - logit(smoothed overall rate).

    Uses only the labels of `rows`, which are always training rows.
    """
    y = rows[LABEL_COL]
    per_step = y.groupby(rows[TIME_COL]).agg(["sum", "size"])
    step_rate = (per_step["sum"] + 0.5) / (per_step["size"] + 1)
    overall_rate = (y.sum() + 0.5) / (len(y) + 1)
    return (logit(step_rate) - logit(overall_rate)).reindex(rows[TIME_COL]).to_numpy()


def make_lightgbm(seed, balanced=False):
    """LightGBM with the baseline settings from notebooks/03_baseline_models.ipynb (kept fixed: tuning did not help on the leaderboard)."""
    return lgb.LGBMClassifier(
        n_estimators=500,           # number of trees
        learning_rate=0.03,         # small steps: more trees, but less overfitting
        num_leaves=31,              # maximum complexity of each tree
        subsample=0.8,              # each tree is built from a random 80% of the rows ...
        subsample_freq=1,           # ... re-drawn for every tree
        colsample_bytree=0.5,       # ... and a random 50% of the features, also drawn once per tree
        class_weight="balanced" if balanced else None,  # balanced: illicit rows count more; None: every row counts the same
        random_state=seed,
        n_jobs=N_THREADS,
        verbose=-1,                 # silence training logs
    )


def fit_lightgbm(rows, cols, seed, balanced=False, offset=None):
    """Fit LightGBM on rows[cols]. `offset` (one value per row, or None) is the fixed starting score."""
    return make_lightgbm(seed, balanced).fit(rows[cols], rows[LABEL_COL], init_score=offset)


# The offsets for all labeled training rows, summarised per time step
per_step = lab[[TIME_COL, LABEL_COL]].assign(offset=step_offset(lab)).groupby(TIME_COL).agg(
    rows=(LABEL_COL, "size"), illicit_rate=(LABEL_COL, "mean"), offset=("offset", "first"))
print(f"Overall illicit rate: {lab[LABEL_COL].mean():.3f}")
print(f"class_weight='balanced' counts each illicit row {(lab[LABEL_COL] == 0).sum() / lab[LABEL_COL].sum():.1f} times as much as a licit row")
print("The three time steps with the lowest and the three with the highest illicit rate:")
per_step.sort_values("illicit_rate").iloc[[0, 1, 2, -3, -2, -1]].round(3)

Overall illicit rate: 0.117
class_weight='balanced' counts each illicit row 7.6 times as much as a licit row
The three time steps with the lowest and the three with the highest illicit rate:


,rows,illicit_rate,offset
time_step,,,
5,1882,0.004,-3.372
1,2147,0.008,-2.778
3,1279,0.009,-2.679
28,284,0.299,1.177
9,778,0.319,1.266
13,809,0.360,1.448


**Takeaway:** The offsets range from −3.372 (step 5) to +1.448 (step 13) on the log-odds scale, a spread of almost five log-odds units.
That is the part of each training row's score the trees no longer need to learn, and that the test predictions do not contain.

## 5. Feature engineering
### 5.1 The present rule
Illicit transactions cluster in the payment graph: only 5.3% of the edges between labeled transactions join an illicit and a licit one, against 14.9% expected at random (`notebooks/01_eda.ipynb`). A transaction's neighbourhood should therefore carry signal.

**The visibility problem.** A training transaction has a row for every neighbour, but a test transaction does not (last check in section 3).
A feature such as "average of the neighbours' features" would be computed over all neighbours in training but over only some neighbours at test time, so the model would learn from a neighbourhood the test set never shows.

**The fix.** A neighbour counts as **present** only if it is a labeled training row (for training transactions) or a `test.csv` row (for test transactions).
This mirrors how `test.csv` appears to have been built (section 3).

**Worked example.** A training transaction A has three neighbours: B (labeled), C (unlabeled, in `train.csv`) and D (in the edge list only).
Counting every neighbour with a row, A sees B and C (2 of 3). Under the present rule, A sees only B (1 of 3), just as a test transaction sees only its `test.csv` neighbours.
The check below measures the effect over all transactions.

In [7]:
# Every (transaction, neighbour) pair, in both edge directions
pairs = pd.concat([
    edges.set_axis(["node", "neighbour"], axis=1),
    edges[["txId2", "txId1"]].set_axis(["node", "neighbour"], axis=1),
]).drop_duplicates()


def neighbour_share(rows, known_ids):
    """Average share of each transaction's neighbours whose id is in `known_ids`."""
    p = pairs[pairs["node"].isin(rows[ID_COL])]
    return p["neighbour"].isin(known_ids).groupby(p["node"]).mean().mean()


present_rows = pd.concat([lab, test])  # the present transactions (the rule above)
pd.DataFrame({  # share of each transaction's neighbours that have a row / that are present
    "share with a row": [neighbour_share(lab, all_ids), neighbour_share(test, all_ids)],
    "share present": [neighbour_share(lab, present_rows[ID_COL]), neighbour_share(test, present_rows[ID_COL])],
}, index=["labeled training rows", "test rows"]).round(3)

,share with a row,share present
labeled training rows,1.000,0.631
test rows,0.609,0.609


**Takeaway:** Counting every neighbour with a row, training transactions see 100% of their neighbours but test transactions only 61%. Under the present rule both see about the same share (63% vs 61%), so the rule removes the mismatch.

I build 27 features with this rule. None of them reads a label value: they only use *which* transactions are present, and that is known for the test period too (the `test.csv` rows). The features are therefore computed in the same way for test rows, without leaking the target.

| Group | Columns | What it describes |
|---|---|---|
| **Presence** (7) | `in_degree`, `out_degree` | How many transactions pay into this one / spend its outputs (full edge list) |
| | `n_in_present`, `n_out_present`, `n_nb_present` | How many of those neighbours are present |
| | `share_present`, `n_nb_absent` | Share of neighbours present (−1 if it has none) and how many are missing |
| **Neighbour profile** (20) | `nb_mean_pc0` … `nb_mean_pc9` | Average of the present neighbours' own features, compressed to 10 principal components |
| | `nb_max_pc0` … `nb_max_pc9` | Largest value among the present neighbours, for the same 10 components |

### 5.2 Graph features
The graph is stored once as sparse 0/1 adjacency matrices, so "sum over my neighbours" becomes one sparse matrix product instead of a Python loop over 234,355 edges.

**Helpers**
- `presence_features(rows, present_ids)`: the 7 degree and presence counts.
- `neighbour_mean_max(rows, present, values, names)`: the mean and max of any per-transaction values over each transaction's **present** neighbours. It builds the 20 neighbour-profile columns here and the 2 neighbours' risk columns in section 5.3.

**Steps in `neighbour_mean_max`:**
1. Place each present transaction's values in an array indexed by node (all other nodes stay 0).
2. Mask the adjacency matrix so that each row keeps only its edges to present neighbours.
3. Mean = (masked adjacency × values) / number of present neighbours.
4. Max = row-wise maximum over each row's present neighbours (`np.maximum.reduceat` over the CSR row segments).
5. Transactions with no present neighbour get NaN, which LightGBM treats as missing.

**Worked example.** If A's two present neighbours have values 0.2 and 0.8 and its third neighbour is absent, A gets `nb_mean` = 0.5 and `nb_max` = 0.8. The absent neighbour is ignored, not counted as 0.

In [8]:
# Number every transaction (including the edge-only ones) 0..n-1
nodes = pd.Index(pd.unique(pd.concat([edges["txId1"], edges["txId2"], train[ID_COL], test[ID_COL]])))
src, dst = nodes.get_indexer(edges["txId1"]), nodes.get_indexer(edges["txId2"])
out_adj = coo_matrix((np.ones(len(src)), (src, dst)), shape=(len(nodes), len(nodes))).tocsr()
out_adj.data[:] = 1          # out_adj[i, j] = 1: transaction j spends an output of i (a repeated edge counts once)
in_adj = out_adj.T.tocsr()   # in_adj[i, j] = 1: transaction j pays into i
nb_adj = out_adj + in_adj
nb_adj.data[:] = 1           # neighbours in either direction, each counted once

PRESENCE_COLS = ["in_degree", "out_degree", "n_in_present", "n_out_present", "n_nb_present", "share_present", "n_nb_absent"]


def presence_features(rows, present_ids):
    """Degree and neighbour-presence counts for each transaction in `rows` (same row order)."""
    present = nodes.isin(present_ids).astype(float)  # 1 = present (the rule of section 5.1)
    ix = nodes.get_indexer(rows[ID_COL])
    degree = np.asarray(nb_adj.sum(axis=1)).ravel()[ix]
    n_present = (nb_adj @ present)[ix]
    return pd.DataFrame({
        "in_degree": np.asarray(out_adj.sum(axis=0)).ravel()[ix],   # transactions paying into it
        "out_degree": np.asarray(out_adj.sum(axis=1)).ravel()[ix],  # transactions spending its outputs
        "n_in_present": (in_adj @ present)[ix],
        "n_out_present": (out_adj @ present)[ix],
        "n_nb_present": n_present,
        "share_present": np.where(degree > 0, n_present / np.maximum(degree, 1), -1.0),  # -1 = no neighbours
        "n_nb_absent": degree - n_present,
    }, index=rows.index)


def neighbour_mean_max(rows, present, values, names):
    """Mean and max of `values` over the PRESENT neighbours of each transaction in `rows`.

    `values` has one row per transaction in `present` (same order) and one column per name in `names`.
    Transactions with no present neighbour get NaN, which LightGBM handles as "missing".
    """
    node_values = np.zeros((len(nodes), values.shape[1]))
    is_present = np.zeros(len(nodes))
    pix = nodes.get_indexer(present[ID_COL])
    node_values[pix] = values
    is_present[pix] = 1

    # Keep only the edges that lead to a present neighbour, one matrix row per transaction in `rows`
    nb = nb_adj[nodes.get_indexer(rows[ID_COL])].multiply(is_present[None, :]).tocsr()
    nb.eliminate_zeros()
    count = np.asarray(nb.sum(axis=1)).ravel()

    mean = (nb @ node_values) / np.maximum(count, 1)[:, None]
    mean[count == 0] = np.nan

    maxi = np.full(mean.shape, np.nan)
    has = np.where(count > 0)[0]
    if len(has):  # row-wise max over each row's neighbours (a CSR row's entries are stored together)
        maxi[has] = np.maximum.reduceat(node_values[nb.indices], nb.indptr[:-1][has], axis=0)

    cols = [f"nb_mean_{n}" for n in names] + [f"nb_max_{n}" for n in names]
    return pd.DataFrame(np.hstack([mean, maxi]), columns=cols, index=rows.index)

**Why principal components?** Taking the mean and max of all 93 local features over the neighbours would add 186 highly correlated columns.
PCA first compresses the standardised local features into 10 uncorrelated components, so the model receives a compact summary of what the neighbours look like.

**Design choices**
1. **Fitted on all training-period rows, labeled or not.** PCA uses no labels, so the unlabeled rows add information without any risk of leakage.
2. **Fixed component signs.** A component's sign is arbitrary (scikit-learn versions before 1.5 chose it differently). I make each component's largest weight positive; otherwise a flipped component would turn "largest value among the neighbours" into "smallest".
3. **Named solver (`covariance_eigh`).** This is the solver scikit-learn 1.5 and later picks for data of this shape. Naming it makes an older version stop with an error instead of silently giving slightly different components.

In [9]:
N_COMPONENTS = 10
PC_COLS = [f"pc{i}" for i in range(N_COMPONENTS)]


def fit_local_pca(pool):
    """Standardise the 93 local features of `pool` and fit a 10-component PCA with fixed signs (feature values only, no labels)."""
    pca = make_pipeline(StandardScaler(), PCA(N_COMPONENTS, svd_solver="covariance_eigh")).fit(pool[LOCAL_COLS])
    comps = pca[-1].components_
    biggest = np.abs(comps).argmax(axis=1)                               # each component's largest weight ...
    comps *= np.sign(comps[np.arange(N_COMPONENTS), biggest])[:, None]  # ... is made positive (in place, so the fitted PCA changes)
    return pca


pca = fit_local_pca(train)  # all training-period rows, labeled or not
print(f"{N_COMPONENTS} components keep {pca[-1].explained_variance_ratio_.sum():.0%} of the variance of the 93 local features")

10 components keep 67% of the variance of the 93 local features


**Takeaway:** Ten components keep 67% of the variance of the 93 local features, enough for a compact neighbour profile.

`add_graph_features` builds the 27 graph columns for training rows and prediction rows **in one call, under one rule**: present = training rows + prediction rows.
For the submission that means the labeled training rows plus the `test.csv` rows; for a validation fold (section 6) it means the fold's training rows plus its validation rows.

In [10]:
GRAPH_COLS = PRESENCE_COLS + [f"nb_mean_{c}" for c in PC_COLS] + [f"nb_max_{c}" for c in PC_COLS]


def add_graph_features(train_rows, pred_rows, pca):
    """Return copies of train_rows and pred_rows with the 27 graph columns appended (one rule for both)."""
    present = pd.concat([train_rows, pred_rows])
    pcs = np.vstack([pca.transform(train_rows[LOCAL_COLS]), pca.transform(pred_rows[LOCAL_COLS])])
    return [pd.concat([rows,
                       presence_features(rows, present[ID_COL]),
                       neighbour_mean_max(rows, present, pcs, PC_COLS)], axis=1)
            for rows in (train_rows, pred_rows)]


lab_g, test_g = add_graph_features(lab, test, pca)  # _g = with the 27 graph columns

# Because the same rule is used on both sides, the new columns should look alike in training and test
compare = pd.DataFrame({"training rows (mean)": lab_g[PRESENCE_COLS].mean(), "test rows (mean)": test_g[PRESENCE_COLS].mean()})
compare.loc["rows with no present neighbour (share)"] = [lab_g["nb_mean_pc0"].isna().mean(), test_g["nb_mean_pc0"].isna().mean()]
compare.round(3)

,training rows (mean),test rows (mean)
in_degree,1.763,2.018
out_degree,1.102,1.224
n_in_present,0.765,0.830
n_out_present,0.765,0.830
n_nb_present,1.530,1.660
share_present,0.631,0.609
n_nb_absent,1.335,1.582
rows with no present neighbour (share),0.218,0.254


**Takeaway:** The averages are close. Test transactions have slightly more neighbours (in-degree 2.018 vs 1.763), which looks like ordinary drift between the two periods.
A quarter of the test rows (25.4%) have no present neighbour, so their 20 neighbour-profile columns (and the 2 neighbours' risk columns of section 5.3) are missing; only the 7 presence columns describe their neighbourhood.

### 5.3 Neighbours' predicted risk: a two-stage (stacked) model
The principal components describe what the neighbours look like, not how suspicious they are; a model can estimate the latter.
A stage-1 LightGBM on the 165 raw features scores every present transaction, and the **mean and max of these scores over a transaction's present neighbours** become two more features.
A stage-2 LightGBM is then trained on 165 raw + 27 graph + 2 risk = 194 columns.

**Out-of-fold scores, so training rows look like test rows.** A test transaction's neighbours are scored by a stage-1 model that never saw their labels (they are test rows).
For training rows to match, each neighbour's score must also come from a model that never saw that neighbour's label.
Present neighbours always share a time step (section 3), so I split the training **steps** into 5 groups with `GroupKFold`: each group's rows are scored by a stage-1 model trained on the other 4 groups.
Test rows are scored by a stage-1 model trained on all training rows.

**Steps in `two_stage_model` (one seed):**
1. Compute the per-step offsets from the training labels (section 4).
2. Stage 1 for the training rows: 5 LightGBMs on the raw features, each scoring its held-out group of steps.
3. Stage 1 for the prediction rows: 1 LightGBM trained on all training rows.
4. Take the mean and max of these scores (log-odds scale) over each transaction's present neighbours (`neighbour_mean_max`, section 5.2).
5. Stage 2: 1 LightGBM on the 194 columns, with the same offset and no class weight.
6. Predict with the trees only (`init_score` is not added at prediction time).

**Output:** the stage-2 probability of illicit for each prediction row; the labels of the prediction rows are never read.
`two_stage_model` is the whole final model, used unchanged for the validation folds (section 6) and the submission (section 7).

In [11]:
RISK_COLS = ["nb_mean_risk", "nb_max_risk"]
STAGE2_COLS = FEATURE_COLS + GRAPH_COLS + RISK_COLS


def add_neighbour_risk(train_rows, pred_rows, seed, balanced=False, use_offset=True):
    """Stage 1: return copies of train_rows and pred_rows with the 2 neighbour-risk columns appended."""
    offset = step_offset(train_rows) if use_offset else None

    # Out-of-fold scores for the training rows: 5 groups of whole time steps
    oof = np.zeros(len(train_rows))
    for fit_idx, score_idx in GroupKFold(n_splits=5).split(train_rows, groups=train_rows[TIME_COL]):
        part_offset = offset[fit_idx] if use_offset else None
        model = fit_lightgbm(train_rows.iloc[fit_idx], FEATURE_COLS, seed, balanced, part_offset)
        oof[score_idx] = model.predict(train_rows.iloc[score_idx][FEATURE_COLS], raw_score=True)

    # Prediction rows: a stage-1 model trained on all training rows
    full = fit_lightgbm(train_rows, FEATURE_COLS, seed, balanced, offset)
    pred_score = full.predict(pred_rows[FEATURE_COLS], raw_score=True)

    # Mean and max score over the present neighbours (present = training rows + prediction rows)
    present = pd.concat([train_rows, pred_rows])
    scores = np.concatenate([oof, pred_score])[:, None]
    return [pd.concat([rows, neighbour_mean_max(rows, present, scores, ["risk"])], axis=1)
            for rows in (train_rows, pred_rows)]


def two_stage_model(train_rows, pred_rows, seed, balanced=False, use_offset=True):
    """The final model (with the default arguments): stage 1 -> neighbours' risk -> stage 2.

    Returns the stage-2 probability of each prediction row. The labels of pred_rows are never read.
    """
    train_rows, pred_rows = add_neighbour_risk(train_rows, pred_rows, seed, balanced, use_offset)
    offset = step_offset(train_rows) if use_offset else None
    model = fit_lightgbm(train_rows, STAGE2_COLS, seed, balanced, offset)
    return model.predict_proba(pred_rows[STAGE2_COLS])[:, 1]  # trees only: init_score is not added


# Check: how well does each risk feature separate the classes on its own (training rows, out-of-fold)?
lab_r, _ = add_neighbour_risk(lab_g, test_g, SEED)  # _r = with the 2 risk columns; only the training rows are checked
has_nb = lab_r["nb_max_risk"].notna()
print(f"Training rows with a present neighbour: {has_nb.mean():.1%}")
for col in RISK_COLS:
    print(f"AUC of {col} alone on those rows: {roc_auc_score(lab_r.loc[has_nb, LABEL_COL], lab_r.loc[has_nb, col]):.3f}")

Training rows with a present neighbour: 78.2%
AUC of nb_mean_risk alone on those rows: 0.811
AUC of nb_max_risk alone on those rows: 0.784


**Takeaway:** On their own, the neighbours' predicted risks separate illicit from licit transactions well (AUC 0.811 for the mean, 0.784 for the max), although they are built only from the neighbours' scores, never from the transaction's own features.
This makes homophily usable in the test period, where no neighbour has a known label: stage 2 can learn directly that a transaction next to risky-looking transactions is more likely illicit.
The 21.8% of training rows without a present neighbour get NaN, as in section 5.2.

## 6. Validation
The test set lies in the future, so validation must too: every fold trains on earlier time steps and validates on later ones.
A random split lets the model learn from the same or later steps than the ones it is scored on, which inflates the AUC (random forest: 0.9977 with a random split vs 0.9529 with time-based folds; `notebooks/02_validation_setup.ipynb`).

**Design choices**
1. **Four late folds:** train on steps 1 to s−1 and validate on steps s–35 pooled, for s = 25, 27, 29 and 31. Pooling several steps into one AUC mirrors the leaderboard, which ranks all 14 test steps together.
2. **Step 24 is always in training.** Illicit transactions in step 24 look unlike those of earlier steps (a random forest trained on steps 1–20 scores 0.48 AUC on step 24), so folds that validate on step 24 without training on it are decided by that one event.
   `notebooks/05_model_comparison.ipynb` checks this choice: the late folds rank my submitted models much like the public leaderboard (rank correlation +0.70), while the early folds do not (−0.40).
3. **Each fold is rebuilt from its own training period**, exactly like the submission: the PCA is fitted on the `train.csv` rows before step s, the present set is the fold's training rows plus its validation rows (the counterpart of labeled training rows plus `test.csv`), and the offsets and stage-1 models use only the fold's training labels.
4. **Validation labels are dropped before any feature is built** and are used only to compute the AUC. Which validation rows are labeled decides the present set, just as `test.csv` does for the test period.

The table compares the four models of my leaderboard history (column `LB`: each model's public leaderboard AUC; the leaderboard files average 1, 3, 3 and 6 seeds), with one seed (42) to keep the run short.
The cell trains 16 LightGBM models per fold (64 in all), so it takes several minutes.

In [12]:
FOLD_STARTS = [25, 27, 29, 31]  # fold s: train on steps 1..s-1, validate on steps s..35


def baseline(train_rows, pred_rows, seed):
    """LightGBM on the 165 raw features, balanced class weight."""
    return fit_lightgbm(train_rows, FEATURE_COLS, seed, balanced=True).predict_proba(pred_rows[FEATURE_COLS])[:, 1]


def graph_model(train_rows, pred_rows, seed):
    """+ the 27 present-neighbour graph features."""
    cols = FEATURE_COLS + GRAPH_COLS
    return fit_lightgbm(train_rows, cols, seed, balanced=True).predict_proba(pred_rows[cols])[:, 1]


def stacked_balanced(train_rows, pred_rows, seed):
    """+ neighbours' predicted risk (two stages), still with the balanced class weight and no offset."""
    return two_stage_model(train_rows, pred_rows, seed, balanced=True, use_offset=False)


MODELS = {  # name -> (function, its public leaderboard AUC); each row adds one step to the row above
    "baseline": (baseline, 0.94637),
    "+ graph features": (graph_model, 0.95121),
    "+ neighbours' risk": (stacked_balanced, 0.95846),
    "+ step offset, no weight": (two_stage_model, 0.96169),  # the final model: per-step offset, no class weight
}

aucs = {name: {} for name in MODELS}
for s in FOLD_STARTS:
    fit_rows, valid_rows = lab[lab[TIME_COL] < s], lab[lab[TIME_COL] >= s]
    valid_labels = valid_rows[LABEL_COL]
    valid_rows = valid_rows.drop(columns=LABEL_COL)       # from here on, no model or feature can see the validation labels
    fold_pca = fit_local_pca(train[train[TIME_COL] < s])  # PCA from the fold's training period only
    fit_g, valid_g = add_graph_features(fit_rows, valid_rows, fold_pca)
    for name, (model, _) in MODELS.items():
        aucs[name][f"{s}-35"] = roc_auc_score(valid_labels, model(fit_g, valid_g, SEED))

validation = pd.DataFrame(aucs).T  # one row per model, one column per fold (validation steps)
validation["mean"] = validation.mean(axis=1)
validation["LB"] = [lb for _, lb in MODELS.values()]

print("Change in mean validation AUC from each step:")
for previous, name in zip(validation.index[:-1], validation.index[1:]):
    print(f"  {name}: {validation.loc[name, 'mean'] - validation.loc[previous, 'mean']:+.5f}")
validation.round(5)

Change in mean validation AUC from each step:
  + graph features: +0.00044
  + neighbours' risk: +0.00048
  + step offset, no weight: -0.00003


,25-35,27-35,29-35,31-35,mean,LB
baseline,0.99718,0.99785,0.99742,0.99857,0.99776,0.94637
+ graph features,0.99773,0.99831,0.99796,0.99879,0.99820,0.95121
+ neighbours' risk,0.99825,0.99886,0.99857,0.99904,0.99868,0.95846
"+ step offset, no weight",0.99832,0.99885,0.99848,0.99897,0.99865,0.96169


**Takeaway:** The late folds agree with the leaderboard on the first two steps: the graph features and the neighbours' risk each raise the AUC on every fold (+0.00044 and +0.00048 on the mean).
The last step does not register here. The final model **ties** with the neighbours'-risk model (−0.00003 on the mean: better on the first fold, slightly worse on the other three).
Validation therefore could not select the last step; the public leaderboard did (0.95846 → 0.96169). Section 8 sets out why I consider that gain plausible, and what risk remains.

The validation AUCs (about 0.998) sit far above the leaderboard scores (0.946 to 0.962), so I use them only to rank models, not to forecast the leaderboard.

## 7. Train the final model and write the submission
The final model is trained on **all** labeled rows (steps 1–35) and predicts the `test.csv` rows, using the graph features built in section 5.

**Steps:**
1. Run `two_stage_model` once for each of the six seeds in `FINAL_SEEDS`; the seed controls the row and feature sampling in both stages.
2. Average the six predicted probabilities, so the result depends less on one random draw.
3. Check the file against Kaggle's sample submission: same `index` values in the same order, and every probability between 0 and 1 (which also rules out NaN).
4. Write `submissions/e1156682_submission.csv` with `\n` line endings, so the file has the same bytes on Windows and Linux.

The cell trains 42 LightGBM models (7 per seed), so it takes a few minutes.

In [13]:
seed_probs = [two_stage_model(lab_g, test_g, seed) for seed in FINAL_SEEDS]
submission = pd.DataFrame({"index": test["index"], "target": np.mean(seed_probs, axis=0)})

# Checks before uploading: same rows and order as Kaggle's sample file, valid probabilities (no NaN)
assert submission["index"].equals(sample["index"])
assert submission["target"].between(0, 1).all()

SUBMISSIONS_DIR.mkdir(exist_ok=True)
submission.to_csv(SUBMISSIONS_DIR / "e1156682_submission.csv", index=False, lineterminator="\n")  # same bytes on Windows and Linux
print(f"Saved {len(submission):,} predictions to submissions/e1156682_submission.csv")
print(f"Total run time: {(time.time() - START_TIME) / 60:.1f} minutes")
submission.head()

Saved 15,329 predictions to submissions/e1156682_submission.csv
Total run time: 6.7 minutes


,index,target
0,0,0.000025
1,1,0.000419
2,2,0.000091
3,3,0.000050
4,4,0.000014


**Takeaway:** These are the predictions behind my best public score (0.96169).
Outside this notebook I compared them value by value with the file I uploaded and found no difference; the first rows above let anyone compare a rerun with this saved output.

## 8. Summary
| Step | Choice | Why |
|---|---|---|
| Preprocessing | Train on the 31,235 labeled rows; keep the unlabeled rows for the PCA | Only labeled rows can train or score a model |
| Learner | LightGBM with fixed settings, a per-step offset and no class weight | The illicit share per training step ranges from 0.4% to 36.0%; the offset keeps those base rates out of the test scores. Chosen on the public leaderboard; validation tied (section 6) |
| Feature engineering | 165 raw + 27 present-neighbour graph features + 2 neighbours' risk features (stage 1, out-of-fold) | Neighbours carry signal; the present rule and the out-of-fold scores make training rows look like test rows |
| Model | Two-stage (stacked) LightGBM, averaged over 6 seeds | Best public leaderboard score of all my submissions |
| Validation | 4 late time-based folds, each rebuilt from its own training period | Respects time order and compares models, but could not rank the last step |
| Result | Public leaderboard AUC 0.96169 (baseline 0.94637) | |

### Leaderboard history
Public leaderboard (30% of the test rows). Each submission adds one step to the one above.
The files also average different numbers of seeds (column Seeds).

| Submission | Seeds | Public AUC | Change |
|---|---|---|---|
| LightGBM baseline, 165 raw features | 1 | 0.94637 | |
| + 27 present-neighbour graph features | 3 | 0.95121 | +0.00484 |
| + neighbours' predicted risk (two-stage stacking) | 3 | 0.95846 | +0.00725 |
| **+ no class weight and per-step offset (final)** | **6** | **0.96169** | **+0.00323** |

Other leaderboard probes that did not help (lower than, or level with, the model they were built on). `notebooks/05_model_comparison.ipynb` lists every submission.

| Probe | Built on | Public AUC |
|---|---|---|
| More trees, each built from fewer features (1000 trees, 30% of the features per tree) | baseline (0.94637) | 0.94261 |
| The same tuning with the graph features | + graph features (0.95121) | 0.94802 |
| Neighbours' risk over 2 hops as well as 1 | + neighbours' risk (0.95846) | 0.95782 |
| Rank-average blend of three neighbour-risk models | + neighbours' risk (0.95846) | 0.95847 |
| Raising the scores of steps 43–49 against steps 36–42 | + neighbours' risk (0.95846) | 0.95733 |
| Dropping 20 drifting columns | + neighbours' risk (0.95846) | 0.95549 |
| Final model without 6 columns that act as a time proxy | final (0.96169) | 0.95758 |
| Fraud PageRank (from the course's graph lectures): PageRank started from the transactions stage 1 finds risky | final (0.96169) | 0.95715 |
| Community features (from the course's graph lectures): Louvain communities and their average risk | final (0.96169) | 0.96136 |

Six earlier probes (self-training, a regularised bag of LightGBMs, a random forest, density features, chain pooling and smooth models) all scored below the baseline's 0.94637.
Other graph measures from the lectures (centrality and clustering) tied with the final model in research runs on the late folds, so they were not submitted.

### The two final submissions
Kaggle ranks each team on the private leaderboard (the other 70% of the test rows) with the better of its two selected files, so the second file should be strong but different.
1. **This notebook's file** (`probe_nocw_stepoff.csv` on Kaggle, public 0.96169): my best public score.
2. **The neighbours'-risk model** (`probe_neighbour_risk.csv`, public 0.95846, rebuilt by `notebooks/04_graph_features.ipynb`): the best model without the last step. If the last step's gain is noise in the public 30%, this file does not share it.
   The community-features probe scored higher (0.96136), but it is the final model plus one feature, so it would most likely fail in the same way.

### How the last step was chosen
On the late folds the final model ties with the 0.95846 model (−0.00003 on the mean, section 6), so validation alone would not have picked it.
I chose it on the **public leaderboard**, where it gained +0.00323. Why I consider the gain plausible and not just luck:
- **It is aimed at the shift into the test period.** The training steps' illicit shares range from 0.4% to 36.0% (section 3), and the test steps 36–49 have their own, unknown shares.
  The offset keeps the training steps' base rates out of the test scores, and the trees are fitted to what separates illicit from licit within a step.
  I expected this to matter more for the test steps than for my folds, but no output here shows it: the folds' validation steps also differ in illicit share (for example step 28 at 29.9%, section 4).
- **The two changes work together.** Each one alone gained much less on the leaderboard: the per-step offset with the class weight kept scored 0.95933 (+0.00087),
  and no class weight with one constant offset for all steps (so no per-step rates) scored 0.95888 (+0.00042). Together they gained +0.00323, more than the two added up.
  The final file and both of these probes average 6 seeds, while the 0.95846 file averages 3, so part of each gain may come from the extra seeds.
  With the same 6 seeds the two probes gained much less, so the extra seeds alone are unlikely to explain the +0.00323.
  No output in this notebook explains why the two changes need each other.
- **The remaining risk.** The public leaderboard uses only 30% of the test rows, so part of the gain may be noise, and the private leaderboard (the other 70%) may rank the models differently.

### Ensembles
The final model is already an ensemble, in two ways:
- **Stacking:** for each seed, six stage-1 LightGBMs (five out-of-fold, one trained on all rows) produce the neighbours' risk features that the stage-2 LightGBM learns from.
- **Seed averaging:** the submission averages six two-stage models that differ only in their random seed (used in both stages), which changes the rows and features each tree is built from.

In research runs outside this notebook, before the last step, I also tried other model types (CatBoost, XGBoost, ExtraTrees and scikit-learn's HistGradientBoosting) and blends of them with the 0.95846 neighbours'-risk model.
None beat that model on the late folds, so I did not submit them.
A rank-average blend of three neighbour-risk models scored 0.95847, level with the single model's 0.95846.

### Limitations
- Validation could not rank the last step; that choice rests on 30% of the test rows.
- The validation AUCs (about 0.998) are far above the leaderboard; they only compare models.
- 25.4% of the test rows have no present neighbour (section 5.2). For them only the degree and presence counts describe the neighbourhood; the neighbour-profile and neighbours' risk features are missing.
- The present rule relies on `test.csv` holding exactly the labeled test-period transactions, which I inferred from the data (section 3). A test file built differently would need a different rule.
- Illicit patterns that only appear after step 35 cannot be learned from the training steps.